In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import matplotlib.gridspec as gridspec
from matplotlib import cm
import matplotlib.ticker as ticker
from scipy.interpolate import griddata
import pandas as pd
plt.rcParams.update({
    'font.size': 22,  
    'font.weight': 'bold', 
    'axes.titlesize': 22, 
    'axes.labelsize': 22,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 22,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})
# ================= Results Directory Configuration =================
RESULTS_ROOT = "./pinn_results"  # Modify to your results root directory

# ================= Parameter Configuration =================
K = 50.0  # Strike price

# ================= Results Analysis Functions =================
def extract_params_from_folder(folder_name):
    """Extract alpha, beta, and tau parameters from folder name"""
    # Try to match multiple possible folder naming patterns
    patterns = [
        r'alpha_([\d\.]+)_beta_([\d\.]+)_tau_([\d\.]+)',  # alpha_beta_tau format
        r'beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)',  # beta_alpha_tau format
        r'tau_([\d\.]+)_alpha_([\d\.]+)_beta_([\d\.]+)',  # tau_alpha_beta format
        r'run_alpha_([\d\.]+)_beta_([\d\.]+)_tau_([\d\.]+)',  # with run prefix
        r'run_beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)',  # with run prefix
    ]
    
    for pattern in patterns:
        match = re.search(pattern, folder_name)
        if match:
            # Determine parameter order based on pattern
            if 'alpha' in pattern and pattern.index('alpha') < pattern.index('beta'):
                alpha = float(match.group(1))
                beta = float(match.group(2))
                tau = float(match.group(3))
            elif 'beta' in pattern and pattern.index('beta') < pattern.index('alpha'):
                beta = float(match.group(1))
                alpha = float(match.group(2))
                tau = float(match.group(3))
            elif 'tau' in pattern and pattern.index('tau') < pattern.index('alpha'):
                tau = float(match.group(1))
                alpha = float(match.group(2))
                beta = float(match.group(3))
            else:
                # Default order
                alpha = float(match.group(1))
                beta = float(match.group(2))
                tau = float(match.group(3))
            return alpha, beta, tau
    
    return None, None, None

def find_results_folders():
    """Find all result folders and categorize them"""
    all_folders = []
    
    # Traverse all folders in the results root directory
    for folder in os.listdir(RESULTS_ROOT):
        folder_path = os.path.join(RESULTS_ROOT, folder)
        if os.path.isdir(folder_path):
            alpha, beta, tau = extract_params_from_folder(folder)
            
            if alpha is not None and beta is not None and tau is not None:
                all_folders.append({
                    'path': folder_path,
                    'alpha': alpha,
                    'beta': beta,
                    'tau': tau
                })
    
    return all_folders

def load_results(folder_path):
    """Load result data from folder"""
    results_path = os.path.join(folder_path, "pinn_results.npy")
    if not os.path.exists(results_path):
        print(f"Warning: Results file not found in {folder_path}")
        return None, None, None, None, None
    
    try:
        # Load result data
        results = np.load(results_path)
        if results.shape[1] < 3:
            print(f"Error: Invalid data format in {results_path}")
            return None, None, None, None, None
            
        A_vals = results[:, 0]
        t_vals = results[:, 1]
        C_vals = results[:, 2]
        
        # Get unique A and t values
        unique_A = np.unique(A_vals)
        unique_t = np.unique(t_vals)
        
        # Create mesh
        A_mesh, t_mesh = np.meshgrid(unique_A, unique_t, indexing='ij')
        
        # Reshape predictions to grid shape
        C_grid = C_vals.reshape(len(unique_A), len(unique_t))
        
        return unique_A, unique_t, C_grid, A_mesh, t_mesh
    except Exception as e:
        print(f"Error loading {results_path}: {str(e)}")
        return None, None, None, None, None

def plot_figure3(ax, unique_A, unique_t, C_grid, title):
    """Plot Figure 3: 2D potential surface (A vs C with t contours)"""
    try:
        # Create regular grid for plotting
        A_plot = np.linspace(min(unique_A), max(unique_A), 100)
        C_plot = np.linspace(min(C_grid.flatten()), max(C_grid.flatten()), 100)
        A_flat_plot, C_flat_plot = np.meshgrid(A_plot, C_plot, indexing='ij')
        
        # Prepare original data points for interpolation
        points = np.array([[a, c] for a in unique_A for c in C_grid[unique_A == a]])
        t_values = np.repeat(unique_t, len(unique_A))
        
        # Interpolate to regular grid
        t_grid_interp = griddata(
            points, 
            t_values, 
            (A_flat_plot, C_flat_plot), 
            method='linear'
        )
        
        # Plot contour map
        contour = ax.contourf(A_flat_plot, C_flat_plot, t_grid_interp, 
                             levels=20, cmap='viridis')
        ax.contour(A_flat_plot, C_flat_plot, t_grid_interp, 
                  levels=10, colors='k', linewidths=0.5, alpha=0.5)
        
        # Add colorbar
        cbar = plt.colorbar(contour, ax=ax)
        cbar.set_label("Time (t)")
        
        # Set axis labels and title
        ax.set_xlabel("Asset Price (A)")
        ax.set_ylabel("Option Price (C)")
        ax.set_title(title, fontsize=22)
        ax.grid(True, linestyle="--", alpha=0.3)
        return True
    except Exception as e:
        print(f"Plotting error: {str(e)}")
        return False

def plot_figure4(ax, unique_A, unique_t, C_grid, title):
    """Plot Figure 4: Option price curves at different time points"""
    try:
        # Select representative time points (max 6)
        time_points = np.linspace(min(unique_t), max(unique_t), min(6, len(unique_t)))
        colors = cm.viridis(np.linspace(0, 1, len(time_points)))
        
        # Find nearest index for each time point
        for t_val, color in zip(time_points, colors):
            t_idx = np.argmin(np.abs(unique_t - t_val))
            C_vals = C_grid[:, t_idx]
            
            # Plot curve
            ax.plot(unique_A, C_vals, 
                    label=f"t = {t_val:.1f}", 
                    linewidth=2, 
                    color=color)
        
        # Add strike price reference line
        ax.axvline(x=K, color='r', linestyle='--', alpha=0.7)
        ax.text(K + 1, max(C_grid.flatten())/10, f'K={K}', color='r')
        
        # Set axis labels and title
        ax.set_xlabel("Asset Price (A)")
        ax.set_ylabel("Option Price (C)")
        ax.set_title(title, fontsize=22)
        ax.legend(loc='upper left', fontsize=22)
        ax.grid(True, alpha=0.3)
        return True
    except Exception as e:
        print(f"Plotting error: {str(e)}")
        return False

def create_comparison_figures(folders, scan_type):
    """Create comparison figures: multiple subplots, each for one parameter combination"""
    if not folders:
        print(f"No results found for {scan_type} scan")
        return None, None
    
    print(f"Creating comparison figures for {scan_type} scan with {len(folders)} folders")
    
    # Convert to DataFrame for sorting
    df = pd.DataFrame(folders)
    
    # Sort folders based on scan type
    if scan_type == "Fixed τ=3.0, Varying α and β":
        # Sort by alpha (ascending), then by beta (ascending)
        df = df.sort_values(by=['alpha', 'beta'])
        # Get unique alpha and beta values
        unique_alphas = sorted(df['alpha'].unique())
        unique_betas = sorted(df['beta'].unique())
        nrows = len(unique_alphas)
        ncols = len(unique_betas)
        # Create a mapping for grid positions
        grid_positions = {}
        for i, alpha in enumerate(unique_alphas):
            for j, beta in enumerate(unique_betas):
                grid_positions[(alpha, beta)] = (i, j)
    else:
        # Sort by tau (ascending)

        df = df.sort_values(by=['tau'])
        n = len(df)
        # Fixed 4 columns for tau scan
        ncols = 4
        nrows = 4  # Always single row for tau scan
        # Create a mapping for grid positions
        grid_positions = {}
        for i, tau_val in enumerate(df['tau']):
            grid_positions[tau_val] = i
        # df = df.sort_values(by=['tau'])
        # n = len(df)
        # ncols = min(4, n)
        # nrows = (n + ncols - 1) // ncols
    # ncols2 = min(4, n)
    # nrows2 = (n + ncols2 - 1) // ncols2
    # Create Figure 3 type figure
    fig3 = plt.figure(figsize=(ncols * 8, nrows * 6))
    # fig3.suptitle(f"{scan_type} - Asset Price vs Option Price", fontsize=22, y=0.98)
    
    # Create Figure 4 type figure
    fig4 = plt.figure(figsize=(ncols * 10, nrows * 6))
    # fig4.suptitle(f"{scan_type} - Option Price Curves at Different Times", fontsize=22, y=0.98)
    
    fig3_success = 0
    fig4_success = 0
    
    # Iterate through sorted folders
    for idx, folder_info in df.iterrows():
        folder_path = folder_info['path']
        alpha = folder_info['alpha']
        beta = folder_info['beta']
        tau = folder_info['tau']
        
        # Load results
        unique_A, unique_t, C_grid, A_mesh, t_mesh = load_results(folder_path)
        if unique_A is None:
            continue
        
        # Create label for current parameters
        param_label = f"α={alpha}, β={beta}, τ={tau}"
        
        # Determine subplot position
        if scan_type == "Fixed τ=3.0, Varying α and β":
            # Get grid position from mapping
            i, j = grid_positions[(alpha, beta)]
            pos = i * ncols + j + 1
        else:
            # For tau scan, position is based on sorted index
            pos = idx + 1  # Linear index for tau scan
        
        # ===== Add subplot to Figure 3 =====
        ax3 = fig3.add_subplot(nrows, ncols, pos)
        if plot_figure3(ax3, unique_A, unique_t, C_grid, param_label):
            fig3_success += 1
        
        # ===== Add subplot to Figure 4 =====
        ax4 = fig4.add_subplot(nrows, ncols, pos)
        if plot_figure4(ax4, unique_A, unique_t, C_grid, param_label):
            fig4_success += 1
    
    # Add row and column labels for alpha-beta scan
    if scan_type == "Fixed τ=3.0, Varying α and β" and fig3_success > 0:
        # Add row labels (alpha values) to the left of the grid
        for i, alpha_val in enumerate(unique_alphas):
            fig3.text(0.01, 0.95 - i*0.9/nrows, f"α={alpha_val}", 
                     rotation='vertical', fontsize=12, va='center')
            fig4.text(0.01, 0.95 - i*0.9/nrows, f"α={alpha_val}", 
                     rotation='vertical', fontsize=12, va='center')
        
        # Add column labels (beta values) above the grid
        for j, beta_val in enumerate(unique_betas):
            fig3.text(0.1 + j*0.8/ncols, 0.97, f"β={beta_val}", 
                     fontsize=12, ha='center')
            fig4.text(0.1 + j*0.8/ncols, 0.97, f"β={beta_val}", 
                     fontsize=12, ha='center')
    
    # Adjust layout
    if fig3_success > 0:
        fig3.tight_layout(rect=[0.05 if scan_type == "Fixed τ=3.0, Varying α and β" else 0, 
                                0, 1, 0.96])  # Leave space for title
    else:
        plt.close(fig3)
        fig3 = None
    
    if fig4_success > 0:
        fig4.tight_layout(rect=[0.05 if scan_type == "Fixed τ=3.0, Varying α and β" else 0, 
                                0, 1, 0.96])  # Leave space for title
    else:
        plt.close(fig4)
        fig4 = None
    
    return fig3, fig4

def filter_folders_by_scan_type(all_folders):
    """Filter folders by scan type"""
    # Fixed tau scan (tau=3.0)
    fixed_tau_folders = [f for f in all_folders if f['tau'] == 3.0]
    
    # Fixed alpha and beta scan (alpha=1.2, beta=0.5)
    fixed_alpha_beta_folders = [
        f for f in all_folders 
        if f['alpha'] == 1.2 and f['beta'] == 0.5
    ]
    
    return fixed_tau_folders, fixed_alpha_beta_folders

# ================= Main Program =================
if __name__ == "__main__":
    # Find all result folders
    all_folders = find_results_folders()
    print(f"Found {len(all_folders)} result folders")
    
    # Filter folders
    fixed_tau_folders, fixed_alpha_beta_folders = filter_folders_by_scan_type(all_folders)
    
    print(f"Fixed τ scan folders: {len(fixed_tau_folders)}")
    print(f"Fixed α and β scan folders: {len(fixed_alpha_beta_folders)}")
    
    # Create Figure 1 and 2: Fixed tau, scan alpha and beta
    if fixed_tau_folders:
        # Sort by alpha and beta for better organization
        fixed_tau_folders = sorted(fixed_tau_folders, key=lambda x: (x['alpha'], x['beta']))
        fig1, fig2 = create_comparison_figures(fixed_tau_folders, "Fixed τ=3.0, Varying α and β")
        if fig1:
            fig1.savefig("figure1_fixed_tau_figure3.png", dpi=300, bbox_inches='tight')
        if fig2:
            fig2.savefig("figure2_fixed_tau_figure4.png", dpi=300, bbox_inches='tight')
    
    # Create Figure 3 and 4: Fixed alpha and beta, scan tau
    if fixed_alpha_beta_folders:
        # Sort by tau for better organization
        fixed_alpha_beta_folders = sorted(fixed_alpha_beta_folders, key=lambda x: x['tau'])
        fig3, fig4 = create_comparison_figures(fixed_alpha_beta_folders, "Fixed α=1.0 and β=0.5, Varying τ")
        if fig3:
            fig3.savefig("figure3_fixed_alpha_beta_figure3.png", dpi=300, bbox_inches='tight')
        if fig4:
            fig4.savefig("figure4_fixed_alpha_beta_figure4.png", dpi=300, bbox_inches='tight')
    
    # Show all figures
    plt.show()
    print("Analysis completed. Figures saved.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import matplotlib.gridspec as gridspec
from matplotlib import cm
import matplotlib.ticker as ticker
from scipy.interpolate import griddata
import pandas as pd
plt.rcParams.update({
    'font.size': 22,  
    'font.weight': 'bold', 
    'axes.titlesize': 22, 
    'axes.labelsize': 22,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 22,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})
# ================= Results Directory Configuration =================
RESULTS_ROOT = "./pinn_results"  # Modify to your results root directory

# ================= Parameter Configuration =================
K = 50.0  # Strike price

# ================= Results Analysis Functions =================
def extract_params_from_folder(folder_name):
    """Extract alpha, beta, and tau parameters from folder name"""
    # Try to match multiple possible folder naming patterns
    patterns = [
        r'alpha_([\d\.]+)_beta_([\d\.]+)_tau_([\d\.]+)',  # alpha_beta_tau format
        r'beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)',  # beta_alpha_tau format
        r'tau_([\d\.]+)_alpha_([\d\.]+)_beta_([\d\.]+)',  # tau_alpha_beta format
        r'run_alpha_([\d\.]+)_beta_([\d\.]+)_tau_([\d\.]+)',  # with run prefix
        r'run_beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)',  # with run prefix
    ]
    
    for pattern in patterns:
        match = re.search(pattern, folder_name)
        if match:
            # Determine parameter order based on pattern
            if 'alpha' in pattern and pattern.index('alpha') < pattern.index('beta'):
                alpha = float(match.group(1))
                beta = float(match.group(2))
                tau = float(match.group(3))
            elif 'beta' in pattern and pattern.index('beta') < pattern.index('alpha'):
                beta = float(match.group(1))
                alpha = float(match.group(2))
                tau = float(match.group(3))
            elif 'tau' in pattern and pattern.index('tau') < pattern.index('alpha'):
                tau = float(match.group(1))
                alpha = float(match.group(2))
                beta = float(match.group(3))
            else:
                # Default order
                alpha = float(match.group(1))
                beta = float(match.group(2))
                tau = float(match.group(3))
            return alpha, beta, tau
    
    return None, None, None

def find_results_folders():
    """Find all result folders and categorize them"""
    all_folders = []
    
    # Traverse all folders in the results root directory
    for folder in os.listdir(RESULTS_ROOT):
        folder_path = os.path.join(RESULTS_ROOT, folder)
        if os.path.isdir(folder_path):
            alpha, beta, tau = extract_params_from_folder(folder)
            
            if alpha is not None and beta is not None and tau is not None:
                all_folders.append({
                    'path': folder_path,
                    'alpha': alpha,
                    'beta': beta,
                    'tau': tau
                })
    
    return all_folders

def load_results(folder_path):
    """Load result data from folder"""
    results_path = os.path.join(folder_path, "pinn_results.npy")
    if not os.path.exists(results_path):
        print(f"Warning: Results file not found in {folder_path}")
        return None, None, None, None, None
    
    try:
        # Load result data
        results = np.load(results_path)
        if results.shape[1] < 3:
            print(f"Error: Invalid data format in {results_path}")
            return None, None, None, None, None
            
        A_vals = results[:, 0]
        t_vals = results[:, 1]
        C_vals = results[:, 2]
        
        # Get unique A and t values
        unique_A = np.unique(A_vals)
        unique_t = np.unique(t_vals)
        
        # Create mesh
        A_mesh, t_mesh = np.meshgrid(unique_A, unique_t, indexing='ij')
        
        # Reshape predictions to grid shape
        C_grid = C_vals.reshape(len(unique_A), len(unique_t))
        
        return unique_A, unique_t, C_grid, A_mesh, t_mesh
    except Exception as e:
        print(f"Error loading {results_path}: {str(e)}")
        return None, None, None, None, None

def plot_figure3(ax, unique_A, unique_t, C_grid, title):
    """Plot Figure 3: 2D potential surface (A vs C with t contours)"""
    try:
        # Create regular grid for plotting
        A_plot = np.linspace(min(unique_A), max(unique_A), 100)
        C_plot = np.linspace(min(C_grid.flatten()), max(C_grid.flatten()), 100)
        A_flat_plot, C_flat_plot = np.meshgrid(A_plot, C_plot, indexing='ij')
        
        # Prepare original data points for interpolation
        points = np.array([[a, c] for a in unique_A for c in C_grid[unique_A == a]])
        t_values = np.repeat(unique_t, len(unique_A))
        
        # Interpolate to regular grid
        t_grid_interp = griddata(
            points, 
            t_values, 
            (A_flat_plot, C_flat_plot), 
            method='linear'
        )
        
        # Plot contour map
        contour = ax.contourf(A_flat_plot, C_flat_plot, t_grid_interp, 
                             levels=20, cmap='viridis')
        ax.contour(A_flat_plot, C_flat_plot, t_grid_interp, 
                  levels=10, colors='k', linewidths=0.5, alpha=0.5)
        
        # Add colorbar
        cbar = plt.colorbar(contour, ax=ax)
        cbar.set_label("Time (t)")
        
        # Set axis labels and title
        ax.set_xlabel("Asset Price (A)")
        ax.set_ylabel("Option Price (C)")
        ax.set_title(title, fontsize=22)
        ax.grid(True, linestyle="--", alpha=0.3)
        return True
    except Exception as e:
        print(f"Plotting error: {str(e)}")
        return False

def plot_figure4(ax, unique_A, unique_t, C_grid, title):
    """Plot Figure 4: Option price curves at different time points"""
    try:
        # Select representative time points (max 6)
        time_points = np.linspace(min(unique_t), max(unique_t), min(6, len(unique_t)))
        colors = cm.viridis(np.linspace(0, 1, len(time_points)))
        
        # Find nearest index for each time point
        for t_val, color in zip(time_points, colors):
            t_idx = np.argmin(np.abs(unique_t - t_val))
            C_vals = C_grid[:, t_idx]
            
            # Plot curve
            ax.plot(unique_A, C_vals, 
                    label=f"t = {t_val:.1f}", 
                    linewidth=2, 
                    color=color)
        
        # Add strike price reference line
        ax.axvline(x=K, color='r', linestyle='--', alpha=0.7)
        ax.text(K + 1, max(C_grid.flatten())/10, f'K={K}', color='r')
        
        # Set axis labels and title
        ax.set_xlabel("Asset Price (A)")
        ax.set_ylabel("Option Price (C)")
        ax.set_title(title, fontsize=22)
        ax.legend(loc='upper left', fontsize=22)
        ax.grid(True, alpha=0.3)
        return True
    except Exception as e:
        print(f"Plotting error: {str(e)}")
        return False

def plot_t0_comparison(ax, fixed_alpha_beta_folders):
    """Plot comparison of option prices at t=0 for different tau values"""
    try:
        # Sort folders by tau
        sorted_folders = sorted(fixed_alpha_beta_folders, key=lambda x: x['tau'])
        
        # Create colormap for different tau values
        colors = cm.viridis(np.linspace(0, 1, len(sorted_folders)))
        
        # Plot each tau result
        for folder_info, color in zip(sorted_folders, colors):
            folder_path = folder_info['path']
            tau = folder_info['tau']
            
            # Load results
            unique_A, unique_t, C_grid, _, _ = load_results(folder_path)
            if unique_A is None:
                continue
            
            # Find t=0 index
            t0_idx = np.where(np.isclose(unique_t, 0.0))[0]
            if len(t0_idx) == 0:
                print(f"No t=0 data found for tau={tau}")
                continue
            
            # Get option prices at t=0
            C_t0 = C_grid[:, t0_idx[0]]
            
            # Plot
            ax.plot(unique_A, C_t0, 
                   label=f"τ={tau}", 
                   linewidth=2.5, 
                   color=color)
        
        # Add strike price reference line
        ax.axvline(x=K, color='r', linestyle='--', alpha=0.7)
        ax.text(K + 1, max(unique_A)/20, f'K={K}', color='r')
        
        # Set axis labels and title
        ax.set_xlabel("Asset Price (A)")
        ax.set_ylabel("Option Price at t=0 (C)")
        ax.set_title("Option Prices at t=0 for Different τ Values", fontsize=22)
        ax.legend(loc='lower center', 
                bbox_to_anchor=(0.5, -0.5),  # 稍微降低图例位置
                ncol=min(5, len(sorted_folders)),  # 根据图例项数量调整列数
                frameon=True, 
                framealpha=0.9)
        ax.grid(True, alpha=0.3)
        # ax.set_position([0, 0.05, 1, 0.9])  # 底部留出更多空间给图例
        return True
    except Exception as e:
        print(f"Error in plot_t0_comparison: {str(e)}")
        return False

def create_comparison_figures(folders, scan_type):
    """Create comparison figures: multiple subplots, each for one parameter combination"""
    if not folders:
        print(f"No results found for {scan_type} scan")
        return None, None
    
    print(f"Creating comparison figures for {scan_type} scan with {len(folders)} folders")
    
    # Convert to DataFrame for sorting
    df = pd.DataFrame(folders)
    
    # Sort folders based on scan type
    if scan_type == "Fixed τ=3.0, Varying α and β":
        # Sort by alpha (ascending), then by beta (ascending)
        df = df.sort_values(by=['alpha', 'beta'])
        # Get unique alpha and beta values
        unique_alphas = sorted(df['alpha'].unique())
        unique_betas = sorted(df['beta'].unique())
        nrows = len(unique_alphas)
        ncols = len(unique_betas)
        # Create a mapping for grid positions
        grid_positions = {}
        for i, alpha in enumerate(unique_alphas):
            for j, beta in enumerate(unique_betas):
                grid_positions[(alpha, beta)] = (i, j)
    else:
        # Sort by tau (ascending)
        df = df.sort_values(by=['tau'])
        n = len(df)
        # Fixed 4 columns for tau scan
        ncols = 4
        nrows = 4  # Always single row for tau scan
        # Create a mapping for grid positions
        grid_positions = {}
        for i, tau_val in enumerate(df['tau']):
            grid_positions[tau_val] = i
    
    # Create Figure 3 type figure
    fig3 = plt.figure(figsize=(ncols * 8, nrows * 6))
    
    # Create Figure 4 type figure
    fig4 = plt.figure(figsize=(ncols * 10, nrows * 6))
    
    fig3_success = 0
    fig4_success = 0
    
    # Iterate through sorted folders
    for idx, folder_info in df.iterrows():
        folder_path = folder_info['path']
        alpha = folder_info['alpha']
        beta = folder_info['beta']
        tau = folder_info['tau']
        
        # Load results
        unique_A, unique_t, C_grid, A_mesh, t_mesh = load_results(folder_path)
        if unique_A is None:
            continue
        
        # Create label for current parameters
        param_label = f"α={alpha}, β={beta}, τ={tau}"
        
        # Determine subplot position
        if scan_type == "Fixed τ=3.0, Varying α and β":
            # Get grid position from mapping
            i, j = grid_positions[(alpha, beta)]
            pos = i * ncols + j + 1
        else:
            # For tau scan, position is based on sorted index
            pos = idx + 1  # Linear index for tau scan
        
        # ===== Add subplot to Figure 3 =====
        ax3 = fig3.add_subplot(nrows, ncols, pos)
        if plot_figure3(ax3, unique_A, unique_t, C_grid, param_label):
            fig3_success += 1
        
        # ===== Add subplot to Figure 4 =====
        ax4 = fig4.add_subplot(nrows, ncols, pos)
        if plot_figure4(ax4, unique_A, unique_t, C_grid, param_label):
            fig4_success += 1
    
    # Add row and column labels for alpha-beta scan
    if scan_type == "Fixed τ=3.0, Varying α and β" and fig3_success > 0:
        # Add row labels (alpha values) to the left of the grid
        for i, alpha_val in enumerate(unique_alphas):
            fig3.text(0.01, 0.95 - i*0.9/nrows, f"α={alpha_val}", 
                     rotation='vertical', fontsize=12, va='center')
            fig4.text(0.01, 0.95 - i*0.9/nrows, f"α={alpha_val}", 
                     rotation='vertical', fontsize=12, va='center')
        
        # Add column labels (beta values) above the grid
        for j, beta_val in enumerate(unique_betas):
            fig3.text(0.1 + j*0.8/ncols, 0.97, f"β={beta_val}", 
                     fontsize=12, ha='center')
            fig4.text(0.1 + j*0.8/ncols, 0.97, f"β={beta_val}", 
                     fontsize=12, ha='center')
    
    # Adjust layout
    if fig3_success > 0:
        fig3.tight_layout(rect=[0.05 if scan_type == "Fixed τ=3.0, Varying α and β" else 0, 
                                0, 1, 0.96])  # Leave space for title
    else:
        plt.close(fig3)
        fig3 = None
    
    if fig4_success > 0:
        fig4.tight_layout(rect=[0.05 if scan_type == "Fixed τ=3.0, Varying α and β" else 0, 
                                0, 1, 0.96])  # Leave space for title
    else:
        plt.close(fig4)
        fig4 = None
    
    return fig3, fig4

def filter_folders_by_scan_type(all_folders):
    """Filter folders by scan type"""
    # Fixed tau scan (tau=3.0)
    fixed_tau_folders = [f for f in all_folders if f['tau'] == 3.0]
    
    # Fixed alpha and beta scan (alpha=1.2, beta=0.5)
    fixed_alpha_beta_folders = [
        f for f in all_folders 
        if f['alpha'] == 1.2 and f['beta'] == 0.5
    ]
    
    return fixed_tau_folders, fixed_alpha_beta_folders

# ================= Main Program =================
if __name__ == "__main__":
    # Find all result folders
    all_folders = find_results_folders()
    print(f"Found {len(all_folders)} result folders")
    
    # Filter folders
    fixed_tau_folders, fixed_alpha_beta_folders = filter_folders_by_scan_type(all_folders)
    
    print(f"Fixed τ scan folders: {len(fixed_tau_folders)}")
    print(f"Fixed α and β scan folders: {len(fixed_alpha_beta_folders)}")
    
    # # Create Figure 1 and 2: Fixed tau, scan alpha and beta
    # if fixed_tau_folders:
    #     # Sort by alpha and beta for better organization
    #     fixed_tau_folders = sorted(fixed_tau_folders, key=lambda x: (x['alpha'], x['beta']))
    #     fig1, fig2 = create_comparison_figures(fixed_tau_folders, "Fixed τ=3.0, Varying α and β")
    #     if fig1:
    #         fig1.savefig("figure1_fixed_tau_figure3.png", dpi=300, bbox_inches='tight')
    #     if fig2:
    #         fig2.savefig("figure2_fixed_tau_figure4.png", dpi=300, bbox_inches='tight')
    
    # # Create Figure 3 and 4: Fixed alpha and beta, scan tau
    # if fixed_alpha_beta_folders:
    #     # Sort by tau for better organization
    #     fixed_alpha_beta_folders = sorted(fixed_alpha_beta_folders, key=lambda x: x['tau'])
    #     fig3, fig4 = create_comparison_figures(fixed_alpha_beta_folders, "Fixed α=1.0 and β=0.5, Varying τ")
    #     if fig3:
    #         fig3.savefig("figure3_fixed_alpha_beta_figure3.png", dpi=300, bbox_inches='tight')
    #     if fig4:
    #         fig4.savefig("figure4_fixed_alpha_beta_figure4.png", dpi=300, bbox_inches='tight')
    
    # Create new figure: Option prices at t=0 for different tau values
    if fixed_alpha_beta_folders:
        fig5 = plt.figure(figsize=(12, 8))
        ax5 = fig5.add_subplot(111)
        if plot_t0_comparison(ax5, fixed_alpha_beta_folders):
            # fig5.tight_layout(rect=[0, 0.05, 1, 0.95])
            # plt.tight_layout()  # 底部留出更多空间给图例
            fig5.subplots_adjust(bottom=0.25)  # 底部留出15%的空间给图例
            fig5.savefig("figure5_t0_comparison.png", dpi=300, bbox_inches='tight')
        else:
            plt.close(fig5)
    
    # Show all figures
    plt.show()
    print("Analysis completed. Figures saved.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import matplotlib.gridspec as gridspec
from matplotlib import cm
import matplotlib.ticker as ticker
from scipy.interpolate import griddata
import pandas as pd
plt.rcParams.update({
    'font.size': 22,  
    'font.weight': 'bold', 
    'axes.titlesize': 22, 
    'axes.labelsize': 22,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 22,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})
# ================= Results Directory Configuration =================
RESULTS_ROOT = "./pinn_results"  # Modify to your results root directory

# ================= Parameter Configuration =================
K = 50.0  # Strike price

# ================= Results Analysis Functions =================
def extract_params_from_folder(folder_name):
    """Extract alpha, beta, and tau parameters from folder name"""
    # Try to match multiple possible folder naming patterns
    patterns = [
        r'alpha_([\d\.]+)_beta_([\d\.]+)_tau_([\d\.]+)',  # alpha_beta_tau format
        r'beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)',  # beta_alpha_tau format
        r'tau_([\d\.]+)_alpha_([\d\.]+)_beta_([\d\.]+)',  # tau_alpha_beta format
        r'run_alpha_([\d\.]+)_beta_([\d\.]+)_tau_([\d\.]+)',  # with run prefix
        r'run_beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)',  # with run prefix
    ]
    
    for pattern in patterns:
        match = re.search(pattern, folder_name)
        if match:
            # Determine parameter order based on pattern
            if 'alpha' in pattern and pattern.index('alpha') < pattern.index('beta'):
                alpha = float(match.group(1))
                beta = float(match.group(2))
                tau = float(match.group(3))
            elif 'beta' in pattern and pattern.index('beta') < pattern.index('alpha'):
                beta = float(match.group(1))
                alpha = float(match.group(2))
                tau = float(match.group(3))
            elif 'tau' in pattern and pattern.index('tau') < pattern.index('alpha'):
                tau = float(match.group(1))
                alpha = float(match.group(2))
                beta = float(match.group(3))
            else:
                # Default order
                alpha = float(match.group(1))
                beta = float(match.group(2))
                tau = float(match.group(3))
            return alpha, beta, tau
    
    return None, None, None

def find_results_folders():
    """Find all result folders and categorize them"""
    all_folders = []
    
    # Traverse all folders in the results root directory
    for folder in os.listdir(RESULTS_ROOT):
        folder_path = os.path.join(RESULTS_ROOT, folder)
        if os.path.isdir(folder_path):
            alpha, beta, tau = extract_params_from_folder(folder)
            
            if alpha is not None and beta is not None and tau is not None:
                all_folders.append({
                    'path': folder_path,
                    'alpha': alpha,
                    'beta': beta,
                    'tau': tau
                })
    
    return all_folders

def load_results(folder_path):
    """Load result data from folder"""
    results_path = os.path.join(folder_path, "pinn_results.npy")
    if not os.path.exists(results_path):
        print(f"Warning: Results file not found in {folder_path}")
        return None, None, None, None, None
    
    try:
        # Load result data
        results = np.load(results_path)
        if results.shape[1] < 3:
            print(f"Error: Invalid data format in {results_path}")
            return None, None, None, None, None
            
        A_vals = results[:, 0]
        t_vals = results[:, 1]
        C_vals = results[:, 2]
        
        # Get unique A and t values
        unique_A = np.unique(A_vals)
        unique_t = np.unique(t_vals)
        
        # Create mesh
        A_mesh, t_mesh = np.meshgrid(unique_A, unique_t, indexing='ij')
        
        # Reshape predictions to grid shape
        C_grid = C_vals.reshape(len(unique_A), len(unique_t))
        
        return unique_A, unique_t, C_grid, A_mesh, t_mesh
    except Exception as e:
        print(f"Error loading {results_path}: {str(e)}")
        return None, None, None, None, None

def plot_figure3(ax, unique_A, unique_t, C_grid, title):
    """Plot Figure 3: 2D potential surface (A vs C with t contours)"""
    try:
        # Create regular grid for plotting
        A_plot = np.linspace(min(unique_A), max(unique_A), 100)
        C_plot = np.linspace(min(C_grid.flatten()), max(C_grid.flatten()), 100)
        A_flat_plot, C_flat_plot = np.meshgrid(A_plot, C_plot, indexing='ij')
        
        # Prepare original data points for interpolation
        points = np.array([[a, c] for a in unique_A for c in C_grid[unique_A == a]])
        t_values = np.repeat(unique_t, len(unique_A))
        
        # Interpolate to regular grid
        t_grid_interp = griddata(
            points, 
            t_values, 
            (A_flat_plot, C_flat_plot), 
            method='linear'
        )
        
        # Plot contour map
        contour = ax.contourf(A_flat_plot, C_flat_plot, t_grid_interp, 
                             levels=20, cmap='viridis')
        ax.contour(A_flat_plot, C_flat_plot, t_grid_interp, 
                  levels=10, colors='k', linewidths=0.5, alpha=0.5)
        
        # Add colorbar
        cbar = plt.colorbar(contour, ax=ax)
        cbar.set_label("Time (t)")
        
        # Set axis labels and title
        ax.set_xlabel("Asset Price (A)")
        ax.set_ylabel("Option Price (C)")
        ax.set_title(title, fontsize=22)
        ax.grid(True, linestyle="--", alpha=0.3)
        return True
    except Exception as e:
        print(f"Plotting error: {str(e)}")
        return False

def plot_figure4(ax, unique_A, unique_t, C_grid, title):
    """Plot Figure 4: Option price curves at different time points"""
    try:
        # Select representative time points (max 6)
        time_points = np.linspace(min(unique_t), max(unique_t), min(6, len(unique_t)))
        colors = cm.viridis(np.linspace(0, 1, len(time_points)))
        
        # Find nearest index for each time point
        for t_val, color in zip(time_points, colors):
            t_idx = np.argmin(np.abs(unique_t - t_val))
            C_vals = C_grid[:, t_idx]
            
            # Plot curve
            ax.plot(unique_A, C_vals, 
                    label=f"t = {t_val:.1f}", 
                    linewidth=2, 
                    color=color)
        
        # Add strike price reference line
        ax.axvline(x=K, color='r', linestyle='--', alpha=0.7)
        ax.text(K + 1, max(C_grid.flatten())/10, f'K={K}', color='r')
        
        # Set axis labels and title
        ax.set_xlabel("Asset Price (A)")
        ax.set_ylabel("Option Price (C)")
        ax.set_title(title, fontsize=22)
        ax.legend(loc='upper left', fontsize=22)
        ax.grid(True, alpha=0.3)
        return True
    except Exception as e:
        print(f"Plotting error: {str(e)}")
        return False

def plot_t0_comparison(ax, fixed_alpha_beta_folders, selected_taus=None):
    """Plot comparison of option prices at t=0 for selected tau values"""
    try:
        # Sort folders by tau
        sorted_folders = sorted(fixed_alpha_beta_folders, key=lambda x: x['tau'])
        
        # Filter folders if specific taus are selected
        if selected_taus is not None:
            sorted_folders = [f for f in sorted_folders if f['tau'] in selected_taus]
            if not sorted_folders:
                print("No matching tau values found")
                return False
        
        # Create colormap for different tau values
        colors = cm.viridis(np.linspace(0, 1, len(sorted_folders)))
        
        # Plot each selected tau result
        for folder_info, color in zip(sorted_folders, colors):
            folder_path = folder_info['path']
            tau = folder_info['tau']
            
            # Load results
            unique_A, unique_t, C_grid, _, _ = load_results(folder_path)
            if unique_A is None:
                continue
            
            # Find t=0 index
            t0_idx = np.where(np.isclose(unique_t, 0.0))[0]
            if len(t0_idx) == 0:
                print(f"No t=0 data found for tau={tau}")
                continue
            
            # Get option prices at t=0
            C_t0 = C_grid[:, t0_idx[0]]
            
            # Plot
            ax.plot(unique_A, C_t0, 
                   label=f"τ={tau}", 
                   linewidth=2.5, 
                   color=color)
        
        # Add strike price reference line
        ax.axvline(x=K, color='r', linestyle='--', alpha=0.7)
        ax.text(K + 1, max(unique_A)/20, f'K={K}', color='r')
        
        # Set axis labels and title
        ax.set_xlabel("Asset Price (A)")
        ax.set_ylabel("Option Price at t=0 (C)")
        ax.set_title("Option Prices at t=0 for Selected τ Values", fontsize=22)
        ax.legend(loc='upper left', fontsize=22)
        ax.grid(True, alpha=0.3)
        
        return True
    except Exception as e:
        print(f"Error in plot_t0_comparison: {str(e)}")
        return False
def plot_k50_tau_curve(ax, fixed_alpha_beta_folders):
    """Plot option price at A=K=50 for different tau values"""
    try:
        # Sort folders by tau
        sorted_folders = sorted(fixed_alpha_beta_folders, key=lambda x: x['tau'])
        
        tau_values = []
        c_values = []
        
        # Extract data for each tau
        for folder_info in sorted_folders:
            folder_path = folder_info['path']
            tau = folder_info['tau']
            
            # Load results
            unique_A, unique_t, C_grid, _, _ = load_results(folder_path)
            if unique_A is None:
                continue
            
            # Find t=0 index
            t0_idx = np.where(np.isclose(unique_t, 0.0))[0]
            if len(t0_idx) == 0:
                print(f"No t=0 data found for tau={tau}")
                continue
            
            # Get option prices at t=0
            C_t0 = C_grid[:, t0_idx[0]]
            
            # Find A=K=50 index
            a_idx = np.argmin(np.abs(unique_A - K))
            c_at_k = C_t0[a_idx]
            
            tau_values.append(tau)
            c_values.append(c_at_k)
        
        # Plot tau vs C at A=K
        ax.plot(tau_values, c_values, 'o-', linewidth=3, markersize=8)
        
        # Set axis labels and title
        ax.set_xlabel("Time to Maturity (τ)")
        ax.set_ylabel("Option Price at A=K (C)")
        ax.set_title("Option Price at A=K=50 for Different τ Values", fontsize=22)
        ax.grid(True, alpha=0.3)
        
        return True
    except Exception as e:
        print(f"Error in plot_k50_tau_curve: {str(e)}")
        return False

def create_comparison_figures(folders, scan_type):
    """Create comparison figures: multiple subplots, each for one parameter combination"""
    if not folders:
        print(f"No results found for {scan_type} scan")
        return None, None
    
    print(f"Creating comparison figures for {scan_type} scan with {len(folders)} folders")
    
    # Convert to DataFrame for sorting
    df = pd.DataFrame(folders)
    
    # Sort folders based on scan type
    if scan_type == "Fixed τ=3.0, Varying α and β":
        # Sort by alpha (ascending), then by beta (ascending)
        df = df.sort_values(by=['alpha', 'beta'])
        # Get unique alpha and beta values
        unique_alphas = sorted(df['alpha'].unique())
        unique_betas = sorted(df['beta'].unique())
        nrows = len(unique_alphas)
        ncols = len(unique_betas)
        # Create a mapping for grid positions
        grid_positions = {}
        for i, alpha in enumerate(unique_alphas):
            for j, beta in enumerate(unique_betas):
                grid_positions[(alpha, beta)] = (i, j)
    else:
        # Sort by tau (ascending)
        df = df.sort_values(by=['tau'])
        n = len(df)
        # Fixed 4 columns for tau scan
        ncols = 4
        nrows = 4  # Always single row for tau scan
        # Create a mapping for grid positions
        grid_positions = {}
        for i, tau_val in enumerate(df['tau']):
            grid_positions[tau_val] = i
    
    # Create Figure 3 type figure
    fig3 = plt.figure(figsize=(ncols * 8, nrows * 6))
    
    # Create Figure 4 type figure
    fig4 = plt.figure(figsize=(ncols * 10, nrows * 6))
    
    fig3_success = 0
    fig4_success = 0
    
    # Iterate through sorted folders
    for idx, folder_info in df.iterrows():
        folder_path = folder_info['path']
        alpha = folder_info['alpha']
        beta = folder_info['beta']
        tau = folder_info['tau']
        
        # Load results
        unique_A, unique_t, C_grid, A_mesh, t_mesh = load_results(folder_path)
        if unique_A is None:
            continue
        
        # Create label for current parameters
        param_label = f"α={alpha}, β={beta}, τ={tau}"
        
        # Determine subplot position
        if scan_type == "Fixed τ=3.0, Varying α and β":
            # Get grid position from mapping
            i, j = grid_positions[(alpha, beta)]
            pos = i * ncols + j + 1
        else:
            # For tau scan, position is based on sorted index
            pos = idx + 1  # Linear index for tau scan
        
        # ===== Add subplot to Figure 3 =====
        ax3 = fig3.add_subplot(nrows, ncols, pos)
        if plot_figure3(ax3, unique_A, unique_t, C_grid, param_label):
            fig3_success += 1
        
        # ===== Add subplot to Figure 4 =====
        ax4 = fig4.add_subplot(nrows, ncols, pos)
        if plot_figure4(ax4, unique_A, unique_t, C_grid, param_label):
            fig4_success += 1
    
    # Add row and column labels for alpha-beta scan
    if scan_type == "Fixed τ=3.0, Varying α and β" and fig3_success > 0:
        # Add row labels (alpha values) to the left of the grid
        for i, alpha_val in enumerate(unique_alphas):
            fig3.text(0.01, 0.95 - i*0.9/nrows, f"α={alpha_val}", 
                     rotation='vertical', fontsize=12, va='center')
            fig4.text(0.01, 0.95 - i*0.9/nrows, f"α={alpha_val}", 
                     rotation='vertical', fontsize=12, va='center')
        
        # Add column labels (beta values) above the grid
        for j, beta_val in enumerate(unique_betas):
            fig3.text(0.1 + j*0.8/ncols, 0.97, f"β={beta_val}", 
                     fontsize=12, ha='center')
            fig4.text(0.1 + j*0.8/ncols, 0.97, f"β={beta_val}", 
                     fontsize=12, ha='center')
    
    # Adjust layout
    if fig3_success > 0:
        fig3.tight_layout(rect=[0.05 if scan_type == "Fixed τ=3.0, Varying α and β" else 0, 
                                0, 1, 0.96])  # Leave space for title
    else:
        plt.close(fig3)
        fig3 = None
    
    if fig4_success > 0:
        fig4.tight_layout(rect=[0.05 if scan_type == "Fixed τ=3.0, Varying α and β" else 0, 
                                0, 1, 0.96])  # Leave space for title
    else:
        plt.close(fig4)
        fig4 = None
    
    return fig3, fig4

def filter_folders_by_scan_type(all_folders):
    """Filter folders by scan type"""
    # Fixed tau scan (tau=3.0)
    fixed_tau_folders = [f for f in all_folders if f['tau'] == 3.0]
    
    # Fixed alpha and beta scan (alpha=1.2, beta=0.5)
    fixed_alpha_beta_folders = [
        f for f in all_folders 
        if f['alpha'] == 1.2 and f['beta'] == 0.5
    ]
    
    return fixed_tau_folders, fixed_alpha_beta_folders

# ================= Main Program =================
if __name__ == "__main__":
    # Find all result folders
    all_folders = find_results_folders()
    print(f"Found {len(all_folders)} result folders")
    
    # Filter folders
    fixed_tau_folders, fixed_alpha_beta_folders = filter_folders_by_scan_type(all_folders)
    
    print(f"Fixed τ scan folders: {len(fixed_tau_folders)}")
    print(f"Fixed α and β scan folders: {len(fixed_alpha_beta_folders)}")
    
    # Create Figure 1 and 2: Fixed tau, scan alpha and beta
    if fixed_tau_folders:
        # Sort by alpha and beta for better organization
        fixed_tau_folders = sorted(fixed_tau_folders, key=lambda x: (x['alpha'], x['beta']))
        fig1, fig2 = create_comparison_figures(fixed_tau_folders, "Fixed τ=3.0, Varying α and β")
        if fig1:
            fig1.savefig("figure1_fixed_tau_figure3.png", dpi=300, bbox_inches='tight')
        if fig2:
            fig2.savefig("figure2_fixed_tau_figure4.png", dpi=300, bbox_inches='tight')
    
    # Create Figure 3 and 4: Fixed alpha and beta, scan tau
    if fixed_alpha_beta_folders:
        # Sort by tau for better organization
        fixed_alpha_beta_folders = sorted(fixed_alpha_beta_folders, key=lambda x: x['tau'])
        fig3, fig4 = create_comparison_figures(fixed_alpha_beta_folders, "Fixed α=1.0 and β=0.5, Varying τ")
        if fig3:
            fig3.savefig("figure3_fixed_alpha_beta_figure3.png", dpi=300, bbox_inches='tight')
        if fig4:
            fig4.savefig("figure4_fixed_alpha_beta_figure4.png", dpi=300, bbox_inches='tight')
    
    # Create new figure: Option prices at t=0 for different tau values
    if fixed_alpha_beta_folders:
        fig5 = plt.figure(figsize=(12, 8))
        ax5 = fig5.add_subplot(111)
        selected_taus=[0.5, 1.0,1.5,2.0,2.5,3.0, 3.5]
        if plot_t0_comparison(ax5, fixed_alpha_beta_folders,selected_taus):
            fig5.tight_layout()
            # fig5.subplots_adjust(bottom=0.15)  # 底部留出15%的空间给图例
            fig5.savefig("figure5_t0_comparison.png", dpi=300, bbox_inches='tight')
        else:
            plt.close(fig5)
    
    # Create new figure: Option price at A=K=50 for different tau values
    if fixed_alpha_beta_folders:
        fig6 = plt.figure(figsize=(12, 8))
        ax6 = fig6.add_subplot(111)
        if plot_k50_tau_curve(ax6, fixed_alpha_beta_folders):
            fig6.tight_layout()
            fig6.savefig("figure6_k50_tau_curve.png", dpi=300, bbox_inches='tight')
        else:
            plt.close(fig6)
    
    # Show all figures
    plt.show()
    print("Analysis completed. Figures saved.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# 设置结果根目录
RESULTS_ROOT = "./pinn_results_alphabeta1"

# 设置固定的t和A值
t_fixed = 3.0
A_fixed = 80.0

# 获取所有结果文件夹
folders = [f for f in os.listdir(RESULTS_ROOT) if os.path.isdir(os.path.join(RESULTS_ROOT, f))]

# 提取参数和对应的C值
alpha_values = []
beta_values = []
C_values = []

print("Processing result folders...")
for folder in folders:
    # 从文件夹名中提取参数
    match = re.search(r'beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)', folder)
    if match:
        beta = float(match.group(1))
        alpha = float(match.group(2))
        tau = float(match.group(3))
        
        # 读取结果文件
        results_path = os.path.join(RESULTS_ROOT, folder, "pinn_results.npy")
        if os.path.exists(results_path):
            try:
                data = np.load(results_path)
                # 提取A、t和C值
                A_vals = data[:, 0]
                t_vals = data[:, 1]
                C_vals = data[:, 2]
                
                # 找到最接近固定t和A的点
                t_idx = np.argmin(np.abs(t_vals - t_fixed))
                A_idx = np.argmin(np.abs(A_vals - A_fixed))
                
                # 计算距离权重
                t_dist = np.abs(t_vals[t_idx] - t_fixed)
                A_dist = np.abs(A_vals[A_idx] - A_fixed)
                
                # 如果距离太大，跳过这个点
                if t_dist > 0.1 or A_dist > 1.0:
                    print(f"Warning: No close point found for {folder}, skipping...")
                    continue
                
                # 获取C值
                C_val = C_vals[t_idx]
                
                # 存储参数和C值
                alpha_values.append(alpha)
                beta_values.append(beta)
                C_values.append(C_val)
                
            except Exception as e:
                print(f"Error processing {folder}: {e}")
        else:
            print(f"Results file not found in {folder}")

# 转换为numpy数组
alpha_arr = np.array(alpha_values)
beta_arr = np.array(beta_values)
C_arr = np.array(C_values)

print(f"Processed {len(alpha_arr)} data points")

# 创建二维网格用于插值
alpha_unique = np.unique(alpha_arr)
beta_unique = np.unique(beta_arr)

# 创建网格
alpha_grid, beta_grid = np.meshgrid(alpha_unique, beta_unique)

# 插值C值到网格上
from scipy.interpolate import griddata
C_grid = griddata((alpha_arr, beta_arr), C_arr, (alpha_grid, beta_grid), method='cubic')

# 创建自定义颜色映射
colors = [(0, 0, 1), (0, 1, 1), (0, 1, 0), (1, 1, 0), (1, 0, 0)]  # 蓝 -> 青 -> 绿 -> 黄 -> 红
n_bins = 100
cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=n_bins)

# 绘制等高线图
plt.figure(figsize=(10, 8))
contour = plt.contourf(alpha_grid, beta_grid, C_grid, 20, cmap=cmap, alpha=0.8)
plt.colorbar(contour, label='Option Price (C)')

# 添加等高线
contour_lines = plt.contour(alpha_grid, beta_grid, C_grid, 10, colors='black', linewidths=0.5)
plt.clabel(contour_lines, inline=True, fontsize=8, fmt='%.2f')

# 添加数据点
# plt.scatter(alpha_arr, beta_arr, c='black', s=10, alpha=0.5)

# 设置标题和标签
plt.title(f'Carbon Option Price (C) at t={t_fixed}, A={A_fixed}')
plt.xlabel('Alpha (α)')
plt.ylabel('Beta (β)')

# 添加网格
plt.grid(True, alpha=0.3)

# 保存图像
plt.savefig(os.path.join(RESULTS_ROOT, f'alpha_beta_contour_t{t_fixed}_A{A_fixed}.png'), dpi=300, bbox_inches='tight')
plt.show()

# 打印统计信息
print(f"Alpha range: {min(alpha_arr):.2f} to {max(alpha_arr):.2f}")
print(f"Beta range: {min(beta_arr):.2f} to {max(beta_arr):.2f}")
print(f"C range: {min(C_arr):.2f} to {max(C_arr):.2f}")

# # 可选：创建3D曲面图
# fig = plt.figure(figsize=(12, 9))
# ax = fig.add_subplot(111, projection='3d')
# surf = ax.plot_surface(alpha_grid, beta_grid, C_grid, cmap=cmap, alpha=0.8)
# fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, label='Option Price (C)')
# ax.set_title(f'3D Surface of Carbon Option Price at t={t_fixed}, A={A_fixed}')
# ax.set_xlabel('Alpha (α)')
# ax.set_ylabel('Beta (β)')
# ax.set_zlabel('Option Price (C)')
# plt.savefig(os.path.join(RESULTS_ROOT, f'alpha_beta_3d_t{t_fixed}_A{A_fixed}.png'), dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import seaborn as sns

# 设置结果根目录
RESULTS_ROOT = "./pinn_results_alphabeta1"

# 设置固定的t值
t_fixed = 1.0

# 选择几个代表性的alpha和beta值
selected_alphas = [0.2, 0.6, 1.0, 1.2]
selected_betas = [0.2, 0.6, 1.0, 1.2]

# 获取所有结果文件夹
folders = [f for f in os.listdir(RESULTS_ROOT) if os.path.isdir(os.path.join(RESULTS_ROOT, f))]

plt.figure(figsize=(12, 10))

# 绘制不同alpha的C-A曲线（固定beta）
plt.subplot(2, 2, 1)
beta_fixed = 0.8
for alpha in selected_alphas:
    # 查找匹配的文件夹
    pattern = f"beta_{beta_fixed}_alpha_{alpha}_tau"
    matched_folders = [f for f in folders if pattern in f]
    
    if matched_folders:
        folder = matched_folders[0]
        results_path = os.path.join(RESULTS_ROOT, folder, "pinn_results.npy")
        data = np.load(results_path)
        
        # 提取t接近固定值的数据
        t_mask = np.isclose(data[:, 1], t_fixed, atol=0.1)
        t_data = data[t_mask]
        
        # 按A排序
        sorted_indices = np.argsort(t_data[:, 0])
        A_sorted = t_data[sorted_indices, 0]
        C_sorted = t_data[sorted_indices, 2]
        
        plt.plot(A_sorted, C_sorted, label=f'α={alpha}')
        
plt.title(f'C vs A (β={beta_fixed}, t={t_fixed})')
plt.xlabel('A (Asset Price)')
plt.ylabel('C (Option Price)')
plt.legend()
plt.grid(True, alpha=0.3)

# 绘制不同beta的C-A曲线（固定alpha）
plt.subplot(2, 2, 2)
alpha_fixed = 0.8
for beta in selected_betas:
    # 查找匹配的文件夹
    pattern = f"beta_{beta}_alpha_{alpha_fixed}_tau"
    matched_folders = [f for f in folders if pattern in f]
    
    if matched_folders:
        folder = matched_folders[0]
        results_path = os.path.join(RESULTS_ROOT, folder, "pinn_results.npy")
        data = np.load(results_path)
        
        # 提取t接近固定值的数据
        t_mask = np.isclose(data[:, 1], t_fixed, atol=0.1)
        t_data = data[t_mask]
        
        # 按A排序
        sorted_indices = np.argsort(t_data[:, 0])
        A_sorted = t_data[sorted_indices, 0]
        C_sorted = t_data[sorted_indices, 2]
        
        plt.plot(A_sorted, C_sorted, label=f'β={beta}')
        
plt.title(f'C vs A (α={alpha_fixed}, t={t_fixed})')
plt.xlabel('A (Asset Price)')
plt.ylabel('C (Option Price)')
plt.legend()
plt.grid(True, alpha=0.3)

# 绘制不同alpha的C-t曲线（固定A和beta）
plt.subplot(2, 2, 3)
beta_fixed = 0.8
A_fixed = 60.0
for alpha in selected_alphas:
    # 查找匹配的文件夹
    pattern = f"beta_{beta_fixed}_alpha_{alpha}_tau"
    matched_folders = [f for f in folders if pattern in f]
    
    if matched_folders:
        folder = matched_folders[0]
        results_path = os.path.join(RESULTS_ROOT, folder, "pinn_results.npy")
        data = np.load(results_path)
        
        # 提取A接近固定值的数据
        A_mask = np.isclose(data[:, 0], A_fixed, atol=1.0)
        A_data = data[A_mask]
        
        # 按t排序
        sorted_indices = np.argsort(A_data[:, 1])
        t_sorted = A_data[sorted_indices, 1]
        C_sorted = A_data[sorted_indices, 2]
        
        plt.plot(t_sorted, C_sorted, label=f'α={alpha}')
        
plt.title(f'C vs t (β={beta_fixed}, A={A_fixed})')
plt.xlabel('t (Time)')
plt.ylabel('C (Option Price)')
plt.legend()
plt.grid(True, alpha=0.3)

# 绘制不同beta的C-t曲线（固定A和alpha）
plt.subplot(2, 2, 4)
alpha_fixed = 0.8
A_fixed = 60.0
for beta in selected_betas:
    # 查找匹配的文件夹
    pattern = f"beta_{beta}_alpha_{alpha_fixed}_tau"
    matched_folders = [f for f in folders if pattern in f]
    
    if matched_folders:
        folder = matched_folders[0]
        results_path = os.path.join(RESULTS_ROOT, folder, "pinn_results.npy")
        data = np.load(results_path)
        
        # 提取A接近固定值的数据
        A_mask = np.isclose(data[:, 0], A_fixed, atol=1.0)
        A_data = data[A_mask]
        
        # 按t排序
        sorted_indices = np.argsort(A_data[:, 1])
        t_sorted = A_data[sorted_indices, 1]
        C_sorted = A_data[sorted_indices, 2]
        
        plt.plot(t_sorted, C_sorted, label=f'β={beta}')
        
plt.title(f'C vs t (α={alpha_fixed}, A={A_fixed})')
plt.xlabel('t (Time)')
plt.ylabel('C (Option Price)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_ROOT, 'parameter_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import re

# 设置结果根目录
RESULTS_ROOT = "./pinn_results_alphabeta1"

# 设置固定的t和A值
t_fixed = 1.0
A_fixed = 60.0

# 获取所有结果文件夹
folders = [f for f in os.listdir(RESULTS_ROOT) if os.path.isdir(os.path.join(RESULTS_ROOT, f))]

# 创建数据框存储结果
data_list = []

for folder in folders:
    # 从文件夹名中提取参数
    match = re.search(r'beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)', folder)
    if match:
        beta = float(match.group(1))
        alpha = float(match.group(2))
        
        # 读取结果文件
        results_path = os.path.join(RESULTS_ROOT, folder, "pinn_results.npy")
        if os.path.exists(results_path):
            try:
                data = np.load(results_path)
                
                # 找到最接近固定t和A的点
                t_idx = np.argmin(np.abs(data[:, 1] - t_fixed))
                A_idx = np.argmin(np.abs(data[:, 0] - A_fixed))
                
                # 获取C值
                C_val = data[t_idx, 2]
                
                # 添加到数据列表
                data_list.append({'alpha': alpha, 'beta': beta, 'C': C_val})
                
            except Exception as e:
                print(f"Error processing {folder}: {e}")

# 创建数据框
df = pd.DataFrame(data_list)

# 计算参数敏感性
alpha_sensitivity = df.groupby('alpha')['C'].mean()
beta_sensitivity = df.groupby('beta')['C'].mean()

# 绘制敏感性分析图
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(alpha_sensitivity.index, alpha_sensitivity.values, 'o-')
plt.title('Sensitivity to Alpha (α)')
plt.xlabel('Alpha (α)')
plt.ylabel('Average Option Price (C)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(beta_sensitivity.index, beta_sensitivity.values, 'o-', color='orange')
plt.title('Sensitivity to Beta (β)')
plt.xlabel('Beta (β)')
plt.ylabel('Average Option Price (C)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_ROOT, 'parameter_sensitivity.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
import re

# 设置结果根目录
RESULTS_ROOT = "./pinn_results_alphabeta1"

# 选择几个参数组合
param_combinations = [
    {'alpha': 0.2, 'beta': 0.2},
    {'alpha': 0.2, 'beta': 1.2},
    {'alpha': 1.2, 'beta': 0.2},
    {'alpha': 1.2, 'beta': 1.2}
]

# 获取所有结果文件夹
folders = [f for f in os.listdir(RESULTS_ROOT) if os.path.isdir(os.path.join(RESULTS_ROOT, f))]

fig = plt.figure(figsize=(15, 12))

for i, params in enumerate(param_combinations):
    alpha = params['alpha']
    beta = params['beta']
    
    # 查找匹配的文件夹
    pattern = f"beta_{beta}_alpha_{alpha}_tau"
    matched_folders = [f for f in folders if pattern in f]
    
    if matched_folders:
        folder = matched_folders[0]
        results_path = os.path.join(RESULTS_ROOT, folder, "pinn_results.npy")
        data = np.load(results_path)
        
        # 提取唯一的时间点
        unique_t = np.unique(data[:, 1])
        unique_A = np.unique(data[:, 0])
        
        # 创建网格
        T_grid, A_grid = np.meshgrid(unique_t, unique_A)
        
        # 重塑C值为网格形式
        C_grid = data[:, 2].reshape(len(unique_A), len(unique_t))
        
        # 创建3D子图
        ax = fig.add_subplot(2, 2, i+1, projection='3d')
        surf = ax.plot_surface(A_grid, T_grid, C_grid, cmap='viridis', alpha=0.8)
        
        ax.set_title(f'C Surface (α={alpha}, β={beta})')
        ax.set_xlabel('A (Asset Price)')
        ax.set_ylabel('t (Time)')
        ax.set_zlabel('C (Option Price)')
        
        # 添加颜色条
        fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)

plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_ROOT, 'surface_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import pandas as pd

# 设置结果根目录
RESULTS_ROOT = "./pinn_results_alphabeta1"

# 设置固定的t和A值
t_fixed = 0.0
A_fixed = 0.0

# 获取所有结果文件夹
folders = [f for f in os.listdir(RESULTS_ROOT) if os.path.isdir(os.path.join(RESULTS_ROOT, f))]

# 创建数据框存储结果
data_list = []

for folder in folders:
    # 从文件夹名中提取参数
    match = re.search(r'beta_([\d\.]+)_alpha_([\d\.]+)_tau_([\d\.]+)', folder)
    if match:
        beta = float(match.group(1))
        alpha = float(match.group(2))
        tau = float(match.group(3))
        
        # 读取结果文件
        results_path = os.path.join(RESULTS_ROOT, folder, "pinn_results.npy")
        if os.path.exists(results_path):
            try:
                data = np.load(results_path)
                
                # 找到最接近固定t和A的点
                t_idx = np.argmin(np.abs(data[:, 1] - t_fixed))
                A_idx = np.argmin(np.abs(data[:, 0] - A_fixed))
                
                # 获取C值
                C_val = data[t_idx, 2]
                
                # 添加到数据列表
                data_list.append({'alpha': alpha, 'beta': beta, 'tau': tau, 'C': C_val})
                
            except Exception as e:
                print(f"Error processing {folder}: {e}")

# 创建数据框
df = pd.DataFrame(data_list)

# 设置图表样式
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 图1: 固定beta，展示alpha对C的影响
beta_fixed = 0.8  # 选择一个固定的beta值
alpha_subset = df[df['beta'] == beta_fixed].sort_values('alpha')

if not alpha_subset.empty:
    ax1.plot(alpha_subset['alpha'], alpha_subset['C'], 'o-', linewidth=2, markersize=6, color='steelblue')
    ax1.set_xlabel('Alpha (α)', fontsize=12)
    ax1.set_ylabel('Option Price (C)', fontsize=12)
    ax1.set_title(f'Effect of Alpha on C (β={beta_fixed}, t={t_fixed}, A={A_fixed})', fontsize=14)
    ax1.grid(True, alpha=0.7)
    
    # 添加趋势线
    z = np.polyfit(alpha_subset['alpha'], alpha_subset['C'], 1)
    p = np.poly1d(z)
    ax1.plot(alpha_subset['alpha'], p(alpha_subset['alpha']), "r--", alpha=0.7, linewidth=1.5)
    
    # 添加R²值
    from sklearn.metrics import r2_score
    r2 = r2_score(alpha_subset['C'], p(alpha_subset['alpha']))
    ax1.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax1.transAxes, 
             fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
else:
    ax1.text(0.5, 0.5, f'No data for β={beta_fixed}', transform=ax1.transAxes, 
             fontsize=12, ha='center', va='center')

# 图2: 固定alpha，展示beta对C的影响
alpha_fixed = 0.8  # 选择一个固定的alpha值
beta_subset = df[df['alpha'] == alpha_fixed].sort_values('beta')

if not beta_subset.empty:
    ax2.plot(beta_subset['beta'], beta_subset['C'], 'o-', linewidth=2, markersize=6, color='coral')
    ax2.set_xlabel('Beta (β)', fontsize=12)
    ax2.set_ylabel('Option Price (C)', fontsize=12)
    ax2.set_title(f'Effect of Beta on C (α={alpha_fixed}, t={t_fixed}, A={A_fixed})', fontsize=14)
    ax2.grid(True, alpha=0.7)
    
    # 添加趋势线
    z = np.polyfit(beta_subset['beta'], beta_subset['C'], 1)
    p = np.poly1d(z)
    ax2.plot(beta_subset['beta'], p(beta_subset['beta']), "r--", alpha=0.7, linewidth=1.5)
    
    # 添加R²值
    r2 = r2_score(beta_subset['C'], p(beta_subset['beta']))
    ax2.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax2.transAxes, 
             fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
else:
    ax2.text(0.5, 0.5, f'No data for α={alpha_fixed}', transform=ax2.transAxes, 
             fontsize=12, ha='center', va='center')

plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_ROOT, 'alpha_beta_individual_effects.png'), dpi=300, bbox_inches='tight')
plt.show()

# 打印统计信息
print("Alpha effect statistics:")
if not alpha_subset.empty:
    alpha_corr = np.corrcoef(alpha_subset['alpha'], alpha_subset['C'])[0, 1]
    print(f"  Correlation between alpha and C: {alpha_corr:.3f}")
    print(f"  C range for alpha: {alpha_subset['C'].min():.3f} to {alpha_subset['C'].max():.3f}")

print("\nBeta effect statistics:")
if not beta_subset.empty:
    beta_corr = np.corrcoef(beta_subset['beta'], beta_subset['C'])[0, 1]
    print(f"  Correlation between beta and C: {beta_corr:.3f}")
    print(f"  C range for beta: {beta_subset['C'].min():.3f} to {beta_subset['C'].max():.3f}")